# CS 131 Project Assignment 3 — PySpark, Parquet, and DuckDB (Team 9)

Dataset: UCI Online Retail II (both workbook sheets combined into one TSV).

Team roles this sprint: Kanishka Yadav (Data Engineer), Sanjini Shukla (Data Engineer, notebook integrator),
Lipi Himanshukumar Makwana (Product Manager), Kushan Mehta (Data Storyteller).

Run from `pa3/notebooks/` with Java 17 or 21. Set `PA3_DATA_PATH` to the combined TSV if it is not in one of the
default locations checked in Section 0.

## 0. Environment setup and versions

Spark runs in local mode (`local[*]`) on one machine, using the class-style session. The session time zone is UTC
so that timestamp parsing does not depend on the machine's daylight-saving rules.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T
from pathlib import Path
import os
import sys
import platform
import pandas as pd
import duckdb

# Spark workers must use the same Python as this kernel (avoids a version-mismatch error on some machines).
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

spark = (
    SparkSession.builder
    .appName("CS131-PA3-Online-Retail")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("DuckDB version:", duckdb.__version__)
print("Python version:", platform.python_version())
print("Spark master:", spark.sparkContext.master)
print("Spark ran locally:", spark.sparkContext.master.startswith("local"))

### 0a. Paths

Run this notebook from `pa3/notebooks/`. The full dataset is stored outside Git; set `PA3_DATA_PATH` if it is not
in one of the default locations. On IBM:

```bash
export PA3_DATA_PATH=../online_retail_II_combined.tsv
```

In [ ]:
PA3_DIR = Path.cwd().parent
REPO_DIR = PA3_DIR.parent

DATA_PATH = os.environ.get("PA3_DATA_PATH")

if DATA_PATH:
    DATA_PATH = Path(DATA_PATH)
else:
    candidates = [
        REPO_DIR.parent / "online_retail_II_combined.tsv",
        REPO_DIR.parent / "data" / "online_retail_II_combined.tsv",
    ]
    DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Set PA3_DATA_PATH to the combined Online Retail II TSV file."
    )

OUT_DIR = PA3_DIR / "out"
PARQUET_DIR = PA3_DIR / "data" / "parquet" / "online_retail_prepared"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", DATA_PATH)

## 1. Project goal, questions, and data description

*Owner: Lipi (text) — integrated by Sanjini.* Goal, the two questions, row meaning, key fields, important columns,
data types, and units. The data types must match the schema contract in Section 3b.

## 2. Read the data with PySpark

*Owner: Kanishka.*

- **Input:** the combined Online Retail II TSV (both workbook sheets), 1,067,371 transaction rows plus one header.
- **Explicit schema:** all eight columns are read as `StringType`. Casting happens in Section 3, so that cast
  failures can be counted instead of being silently turned into nulls by the reader.
- **Quoting is turned off** (`quote=""`). About 2,500 descriptions contain literal `"` characters. With quoting off,
  Spark splits every line on tabs exactly as PA2's `cut`/`awk` did, so no row can be merged or shifted.
- **`mode="FAILFAST"`** stops the read if any line does not parse into the eight expected fields.
- An empty field is read as `null` (Spark's default `nullValue` is the empty string).

In [ ]:
RAW_COLUMNS = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
]

raw_schema = T.StructType([
    T.StructField(column, T.StringType(), True)
    for column in RAW_COLUMNS
])

raw_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("quote", "")
    .option("mode", "FAILFAST")
    .schema(raw_schema)
    .csv(str(DATA_PATH))
)

raw_rows = raw_df.count()
print("Rows:", raw_rows)
assert raw_rows == 1_067_371, "Expected 1,067,371 rows (PA2: 1,067,372 lines including the header)"

raw_df.printSchema()
raw_df.show(5, truncate=False)

## 3. Select and cast columns

*Owner: Kanishka.*

Columns are renamed to snake_case and cast to their analysis types. Identifier and text columns are kept
**exactly as read**: no trimming, no upper- or lower-casing. This matters for Analysis 1, because PA2 counted
`StockCode` values case-sensitively (for example `85123A` and `85123a` are different codes) and kept one code
with a trailing space (`"47503J "`).

| Column | Source | Type | Unit / meaning |
|---|---|---|---|
| `invoice` | Invoice | string | Invoice number; a leading `C` marks a cancellation |
| `stock_code` | StockCode | string (unchanged) | Product code |
| `description` | Description | string | Product name |
| `quantity` | Quantity | integer | Units of the product on this invoice line |
| `invoice_date` | InvoiceDate | timestamp (`yyyy-MM-dd HH:mm:ss`) | Invoice date and time |
| `price` | Price | double | Price per unit, pounds sterling (GBP) |
| `customer_id` | Customer ID | string | Customer identifier (an ID, not a measure) |
| `country` | Country | string (unchanged) | Customer's country |

**Row meaning:** one row is one invoice line — one product on one invoice.

In [ ]:
typed_df = raw_df.select(
    F.col("Invoice").alias("invoice"),
    F.col("StockCode").alias("stock_code"),
    F.col("Description").alias("description"),
    F.col("Quantity").cast("int").alias("quantity"),
    F.to_timestamp(
        F.col("InvoiceDate"),
        "yyyy-MM-dd HH:mm:ss"
    ).alias("invoice_date"),
    F.col("Price").cast("double").alias("price"),
    F.col("Customer ID").alias("customer_id"),
    F.col("Country").alias("country"),
)

typed_df.printSchema()
typed_df.show(5, truncate=False)

### 3a. Cast failures and missing key values

A **cast failure** is a row whose raw value is present but becomes `null` after casting. A **blank** value is a
non-null string that is empty after trimming; it is counted here but the stored value is not modified.

**Missing-value rules** (applied the same way in PySpark and DuckDB, because DuckDB reads the same Parquet):

| Case | Rule |
|---|---|
| Null or blank `stock_code` | Expected 0. If any appear, Analysis 1 keeps them as their own group (as PA2's `uniq -c` would have); Analysis 2 excludes them. |
| Null or blank `country` | Expected 0. Same rule as `stock_code`. `Unspecified` is a real value and stays as its own country. |
| Missing or invalid `quantity` / `price` | Expected 0 cast failures. Rows are kept; they get `revenue = null` and are flagged by the quality check in Section 5. |
| Missing or invalid `invoice_date` | Expected 0 cast failures. Rows are kept. |
| Missing `customer_id` | Kept and flagged in Section 5. |

In [ ]:
# Cast failure = raw value present, but null after the same cast used in typed_df.
# Blank = non-null string that is empty after trimming (counted only; values are not modified).
checks = raw_df.agg(
    F.sum(F.when(F.col("Quantity").isNotNull() & F.col("Quantity").cast("int").isNull(), 1).otherwise(0)).alias("quantity_cast_failures"),
    F.sum(F.when(F.col("Price").isNotNull() & F.col("Price").cast("double").isNull(), 1).otherwise(0)).alias("price_cast_failures"),
    F.sum(F.when(F.col("InvoiceDate").isNotNull() & F.to_timestamp(F.col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss").isNull(), 1).otherwise(0)).alias("invoice_date_cast_failures"),
    F.sum(F.when(F.col("Quantity").isNull(), 1).otherwise(0)).alias("null_quantity"),
    F.sum(F.when(F.col("Price").isNull(), 1).otherwise(0)).alias("null_price"),
    F.sum(F.when(F.col("InvoiceDate").isNull(), 1).otherwise(0)).alias("null_invoice_date"),
    F.sum(F.when(F.col("StockCode").isNull(), 1).otherwise(0)).alias("null_stock_code"),
    F.sum(F.when(F.trim(F.col("StockCode")) == "", 1).otherwise(0)).alias("blank_stock_code"),
    F.sum(F.when(F.col("Country").isNull(), 1).otherwise(0)).alias("null_country"),
    F.sum(F.when(F.trim(F.col("Country")) == "", 1).otherwise(0)).alias("blank_country"),
    F.sum(F.when(F.col("Customer ID").isNull() | (F.trim(F.col("Customer ID")) == ""), 1).otherwise(0)).alias("missing_customer_id"),
    F.sum(F.when(F.col("StockCode") != F.trim(F.col("StockCode")), 1).otherwise(0)).alias("stock_code_with_edge_whitespace"),
).first().asDict()

for name, value in checks.items():
    print(f"{name:34s} {value:>10,}")

must_be_zero = [
    "quantity_cast_failures", "price_cast_failures", "invoice_date_cast_failures",
    "null_quantity", "null_price", "null_invoice_date",
    "null_stock_code", "blank_stock_code", "null_country", "blank_country",
]
nonzero = {name: checks[name] for name in must_be_zero if checks[name] != 0}
assert not nonzero, f"Unexpected missing or invalid values: {nonzero}"
print("\nAll cast-failure and missing-key checks are zero.")

### 3b. Schema contract for downstream sections

*Handoff from Kanishka to Sanjini and Kushan.* Sections 4–8 add to this DataFrame; Sections 9–12 read the
**reloaded Parquet** from Section 7, not `typed_df`.

| Column | Type | Added in | Notes |
|---|---|---|---|
| `invoice` | string | 3 | Unchanged |
| `stock_code` | string | 3 | **Unchanged — do not trim or change case** |
| `description` | string | 3 | Literal text (quoting was off, so any `"` characters are kept) |
| `quantity` | int | 3 | Units |
| `invoice_date` | timestamp | 3 | Session time zone is UTC |
| `price` | double | 3 | GBP per unit |
| `customer_id` | string | 3 | `null` when missing |
| `country` | string | 3 | Unchanged |
| `revenue` | double | 4 | `quantity * price`, GBP; `null` if either input is `null`; negative for returns |
| `flag_missing_customer` | boolean | 5 | `customer_id IS NULL` |
| `flag_bad_qty_price` | boolean | 5 | `quantity` or `price` is `null` or ≤ 0 |

Parquet location (generated locally, gitignored): `pa3/data/parquet/online_retail_prepared/`.

## 4. Derived columns

*Owner: Kanishka (Day 2). To be completed.*

## 5. Data-quality checks

*Owner: Kanishka (Day 2). To be completed.*

## 6. Write Parquet

*Owner: Kanishka (Day 2). To be completed.*

## 7. Read Parquet back and verify

*Owner: Kanishka (Day 2). To be completed.*

## 8. 20% development sample and sample comparison

*Owner: Kanishka (Day 2). To be completed.*

## 9. Analysis 1 — stock-code frequency (PySpark)

*Owner: Sanjini. To be completed.*

## 10. Analysis 2 — top three stock codes per country (window)

*Owner: Sanjini. To be completed.*

## 11. Analysis 1 in DuckDB and row-by-row comparison

*Owner: Sanjini. To be completed.*

## 12. Benchmark

*Owner: Sanjini. To be completed.*

## 13. Chart

*Owner: Kushan (code) — integrated by Sanjini. To be completed.*

## 14. Findings and limitations

*Owner: Kushan (text) — integrated by Sanjini. To be completed.*

## 15. AI-use disclosure

*Owner: Lipi (collects from all members) — integrated by Sanjini. To be completed.*